In [1]:
import pandas as pd

In [5]:
df=pd.read_csv('retail.csv', header=None)
df.columns=['invoice','stockcode','description','quantity','invoicedate','price','customer_id','country','sourcesheet']

C:\Users\PC\AppData\Local\Temp\ipykernel_3184\1614302668.py:1: DtypeWarning: Columns (0: 0) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('retail.csv', header=None)


In [10]:
df['invoicedate']=pd.to_datetime(df['invoicedate'])
df.head()
df.shape
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   invoice      1033036 non-null  object        
 1   stockcode    1033036 non-null  str           
 2   description  1028761 non-null  str           
 3   quantity     1033036 non-null  int64         
 4   invoicedate  1033036 non-null  datetime64[ns]
 5   price        1033036 non-null  float64       
 6   customer_id  797885 non-null   float64       
 7   country      1033036 non-null  str           
 8   sourcesheet  1033036 non-null  str           
dtypes: datetime64[ns](1), float64(2), int64(1), object(1), str(4)
memory usage: 124.3+ MB


In [12]:
# filter and calculate recency,frequency,montary
rfm_data=df[(df['customer_id'].notna()) & (df['quantity']>0)].copy()
snapshot_date=pd.to_datetime('2011-12-10')

rfm=rfm_data.groupby('customer_id').agg(
    recency=('invoicedate',lambda x:(snapshot_date-x.max()).days),
    frequency=('invoice','nunique'),
    monetary=('quantity',lambda x:(x * rfm_data.loc[x.index,'price']).sum())
).reset_index()

In [16]:
# score customers and assign segments
rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5]).astype(int)

def segment_customer(row):
    if row['r_score']>=4 and row['f_score']>=4 and row['m_score']>=4:
        return'champions'
    elif row['r_score']>=3 and row['f_score']>=3:
        return'loyal'
    elif row['r_score']<=2 and row['f_score']>=3:
        return 'at_risk'
    elif row['r_score']>=4 and row['f_score']<=2:
        return 'new'
    elif row['r_score']==3 and row['f_score']<=2:
        return 'needs attention'
    else:
        return 'lost'

rfm['segement']=rfm.apply(segment_customer,axis=1)
rfm['ischurned']=rfm['recency']>90

In [18]:
# merge rfm results back qnto transaction and export 
df=df[(df['customer_id'].notna()) & (df['quantity']>0)].copy()
df['invoicemonth']=df['invoicedate'].dt.to_period('M').dt.to_timestamp()
df['revenue']=df['quantity']*df['price']

df_final=df.merge(
    rfm[['customer_id','recency','frequency','monetary','r_score','m_score','segement','ischurned']],
    on='customer_id',how='left'
)
df_final.to_excel('ecommercedashboarddata.xlsx',index=False)